In [1]:
#imports
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.model_selection import train_test_split
import random

#for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [2]:
#set paths and load train/test files
train_path = "./train.dat"
test_path  = "./test.dat"

#train.dat: load userID, movieID, rating
train_df = pd.read_csv(
    train_path,
    sep = r"\s+",
    header = 0,
    names = ["userID", "movieID", "rating"],
    low_memory = False
)

#test.dat: load userID and movieID only
test_df = pd.read_csv(
    test_path,
    sep = r"\s+",
    header = 0,
    names = ["userID", "movieID"]
)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

train_df.head()

Train shape: (641699, 3)
Test shape : (71299, 2)


,userID,movieID,rating
0,75,3,1.0
1,75,32,4.5
2,75,110,4.0
3,75,163,4.0
4,75,165,4.5


In [3]:
#load optional content files
movie_genres_path      = "./movie_genres.dat"
movie_directors_path   = "./movie_directors.dat"
movie_actors_path      = "./movie_actors.dat"
tags_path              = "./tags.dat"
movie_tags_path        = "./movie_tags.dat"
user_taggedmovies_path = "./user_taggedmovies.dat"

movie_genres = pd.read_csv(movie_genres_path,
                           sep = "\t",
                           header = None,
                           names = ["movieID", "genre"])

movie_directors = pd.read_csv(movie_directors_path,
                              sep = "\t",
                              header = None,
                              names = ["movieID", "directorID", "directorName"],
                              encoding = "latin-1")

movie_actors = pd.read_csv(movie_actors_path,
                           sep = "\t",
                           header = None,
                           names = ["movieID", "actorID", "actorName", "ranking"],
                           encoding = "latin-1",
                           low_memory = False)

tags = pd.read_csv(tags_path,
                   sep = "\t",
                   header = None,
                   names = ["tagID", "tagValue"],
                   encoding = "latin-1")

movie_tags = pd.read_csv(movie_tags_path,
                         sep = "\t",
                         header = None,
                         names = ["movieID", "tagID", "tagWeight"])

user_taggedmovies = pd.read_csv(user_taggedmovies_path,
                                sep = "\t", 
                                header = None,
                                names = ["userID", "movieID", "tagID"])

print("movie_genres:", movie_genres.shape)
print("movie_directors:", movie_directors.shape)
print("movie_actors:", movie_actors.shape)
print("movie_tags:", movie_tags.shape)
print("user_taggedmovies:", user_taggedmovies.shape)


movie_genres: (20810, 2)
movie_directors: (10156, 3)
movie_actors: (231743, 4)
movie_tags: (51796, 3)
user_taggedmovies: (47958, 3)


In [4]:
#create user/movie index maps
user_ids  = pd.concat([train_df["userID"], test_df["userID"]]).unique()
movie_ids = pd.concat([train_df["movieID"], test_df["movieID"]]).unique()
#ID → index
user_to_idx  = {uid: idx for idx, uid in enumerate(sorted(user_ids))}
movie_to_idx = {mid: idx for idx, mid in enumerate(sorted(movie_ids))}
#index → ID
idx_to_user  = {idx: uid for uid, idx in user_to_idx.items()}
idx_to_movie = {idx: mid for mid, idx in movie_to_idx.items()}

num_users  = len(user_to_idx)
num_movies = len(movie_to_idx)

print("Number of users :", num_users)
print("Number of movies:", num_movies)

#add index columns to train and test
train_df["user_idx"]  = train_df["userID"].map(user_to_idx)
train_df["movie_idx"] = train_df["movieID"].map(movie_to_idx)
test_df["user_idx"]   = test_df["userID"].map(user_to_idx)
test_df["movie_idx"]  = test_df["movieID"].map(movie_to_idx)

train_df.head()

Number of users : 2113
Number of movies: 10001


,userID,movieID,rating,user_idx,movie_idx
0,75,3,1.0,0,2
1,75,32,4.5,0,31
2,75,110,4.0,0,104
3,75,163,4.0,0,153
4,75,165,4.5,0,155


In [5]:
#helper functions for rating matrix and RMSE
from scipy.sparse import csr_matrix
#sparse user–movie matrix from known ratings
def sparse_matrix(df, num_users, num_movies):
    rows = df["user_idx"].values
    cols = df["movie_idx"].values
    data = df["rating"].astype(float).values
    s_matrix = csr_matrix((data, (rows, cols)), shape=(num_users, num_movies))
    return s_matrix
    
#dense matrix with fill_value and then known ratings
def dense_matrix(df, num_users, num_movies, fill_value):
    d_matrix = np.full((num_users, num_movies), fill_value, dtype=float)
    rows = df["user_idx"].values
    cols = df["movie_idx"].values
    ratings = df["rating"].astype(float).values
    d_matrix[rows, cols] = ratings
    return d_matrix

def RMSE(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

In [6]:
#split ratings into train and validation sets
train_part, valid_df = train_test_split(
    train_df,
    test_size = 0.1,
    random_state = RANDOM_STATE,
    shuffle = True
)

print("Training ratings:", train_part.shape)
print("Validation ratings:", valid_df.shape)

# sparse matrix for training part
Ratings_train = sparse_matrix(train_part, num_users, num_movies)

Training ratings: (577529, 5)
Validation ratings: (64170, 5)


In [7]:
#baseline biases model
def baseline_biases(df, num_users, num_movies):
    mu = df["rating"].mean()

    #user bias
    user_mean = df.groupby("user_idx")["rating"].mean()
    user_bias = np.zeros(num_users)
    user_bias[user_mean.index.values] = user_mean.values - mu

    #movie bias
    movie_mean = df.groupby("movie_idx")["rating"].mean()
    movie_bias = np.zeros(num_movies)
    movie_bias[movie_mean.index.values] = movie_mean.values - mu

    return mu, user_bias, movie_bias

#prediction: mu + user bias + movie bias
def baseline_predict_array(user_idx_arr, movie_idx_arr, mu, user_bias, movie_bias):
    preds = mu + user_bias[user_idx_arr] + movie_bias[movie_idx_arr]
    return np.clip(preds, 0.0, 5.0)

# compute baseline on training part
mu, user_bias, movie_bias = baseline_biases(train_part, num_users, num_movies)


# dense matrix for NMF, filled with global mean from train_part_df
Ratings_train_nmf = dense_matrix(train_part, num_users, num_movies, fill_value=mu)

# quick RMSE of baseline on validation
user_valid = valid_df["user_idx"].values
movie_valid = valid_df["movie_idx"].values
rating_valid = valid_df["rating"].values

baseline_preds = baseline_predict_array(user_valid, movie_valid, mu, user_bias, movie_bias)
baseline_RMSE = RMSE(rating_valid, baseline_preds)

print("Baseline validation RMSE:", baseline_RMSE)

Baseline validation RMSE: 0.8235129581020387


In [8]:
#NMF latent factor model
def train_nmf_model(R_sparse, num_comp=35, max_iter=500, random_state=RANDOM_STATE):
    nmf = NMF(
        n_components = num_comp,
        init = "random",
        random_state = random_state,
        max_iter = max_iter
    )
    #W: user-factor matrix
    #H: factor-item matrix
    W = nmf.fit_transform(R_sparse)
    H = nmf.components_
    return nmf, W, H

def nmf_full_prediction_matrix(W, H):
    return W @ H

#train NMF on the training portion
nmf_model, W_train, H_train = train_nmf_model(
    Ratings_train_nmf,
    num_comp = 35,#tune
    max_iter = 1000
)

R_hat_train = nmf_full_prediction_matrix(W_train, H_train)

print("W shape:", W_train.shape)
print("H shape:", H_train.shape)

W shape: (2113, 35)
H shape: (35, 10001)


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


In [9]:
#combine baseline and NMF, tune alpha on validation
def combined_pred_arr(user_idx_arr, movie_idx_arr, mu, user_bias, movie_bias, R_hat, alpha=0.7):
    base = mu + user_bias[user_idx_arr] + movie_bias[movie_idx_arr]
    nmf_pred = R_hat[user_idx_arr, movie_idx_arr]
    preds = alpha * nmf_pred + (1.0 - alpha) * base
    return np.clip(preds, 0.0, 5.0)

def evaluate_alpha(valid_df, mu, user_bias, movie_bias, R_hat, alpha_values):
    u_valid = valid_df["user_idx"].values
    m_valid = valid_df["movie_idx"].values
    r_valid = valid_df["rating"].values
    
    results = []
    for a in alpha_values:
        preds = combined_pred_arr(u_valid, m_valid, mu, user_bias, movie_bias, R_hat, alpha=a)
        score = RMSE(r_valid, preds)
        results.append((a, score))
        print(f"alpha={a:.2f} -> RMSE={score:.5f}")
    return results

alpha_grid = [0.0, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
alpha_results = evaluate_alpha(valid_df, mu, user_bias, movie_bias, R_hat_train, alpha_grid)

#best alpha/lowest RSME
best_alpha, best_alpha_RMSE = min(alpha_results, key=lambda x: x[1])
print("\nBest alpha:", best_alpha, "Validation RMSE:", best_alpha_RMSE)

alpha=0.00 -> RMSE=0.82351
alpha=0.20 -> RMSE=0.80982
alpha=0.25 -> RMSE=0.80886
alpha=0.30 -> RMSE=0.80891
alpha=0.35 -> RMSE=0.80997
alpha=0.40 -> RMSE=0.81205
alpha=0.50 -> RMSE=0.81919

Best alpha: 0.25 Validation RMSE: 0.8088605425502559


In [10]:
#retrain on full training data
Ratings_full = sparse_matrix(train_df, num_users, num_movies)

#train NMF on full data - same tuning
nmf_final, W_full, H_full = train_nmf_model(
    Ratings_full,
    num_comp=35,
    max_iter=1000
)

#full NMF prediction matrix
R_hat_full = nmf_full_prediction_matrix(W_full, H_full)

#baseline biases on full training set
mu_full, user_bias_full, movie_bias_full = baseline_biases(train_df, num_users, num_movies)

print("Final model trained.")

Final model trained.


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


In [11]:
#predict ratings for test set user-movie pairs
FINAL_ALPHA = best_alpha  # or set manually, e.g., 0.7

user_test = test_df["user_idx"].values
movie_test = test_df["movie_idx"].values

test_preds = combined_pred_arr(
    user_test,
    movie_test,
    mu_full,
    user_bias_full,
    movie_bias_full,
    R_hat_full,
    alpha=FINAL_ALPHA
)

#check values are within [0,5]
print("Test predictions min:", test_preds.min(), "max:", test_preds.max())

Test predictions min: 0.0 max: 5.0


In [12]:
#save predictions to file
output_path = "./hw4_pred.dat"
np.savetxt(output_path, test_preds, fmt="%.6f")

In [13]:
print("Baseline RMSE:", baseline_RMSE)
print("Final model RMSE:", best_alpha_RMSE)
print("Best alpha:", best_alpha)

Baseline RMSE: 0.8235129581020387
Final model RMSE: 0.8088605425502559
Best alpha: 0.25
